In [ ]:
import polars as pl
import pandas as pd

import sys
sys.path.append('../../04_utils')

from utils import name_finder, low_context_name_finder

In [ ]:
# Define pathings
data_path = '../../01_data/'
out_path = '../../03_output/01_enriched_results/'

In [3]:
#Load data
verbs_df = pl.read_csv(data_path + 'All auxiliary verbs DS.csv', separator= ',' )

verbs_df.head()

Left,KWIC,Right
str,str,str
"""<s> NARRATOR Yes, indeed. </s>…","""are""","""corralled and led to the north…"
"""<s> NARRATOR Yes, indeed. </s>…","""are""","""locked away, to await the end …"
""". </s><s> And in this land, th…","""art""","""new. </s><s> Thou fared well t…"
""", Lordran. </s><s> ALVINA OF T…","""art""","""a strange one! </s><s> Neverth…"
"""new. </s><s> Thou fared well t…","""''m""","""Alvina of the Darkroot Wood. <…"


In [4]:
#Load Character master table
char_master_df = pl.read_csv(data_path + 'das_char_master.csv')\
                   .with_columns(pl.col('Character').str.replace(',','').alias('Character'))

char_master_df.head()

Character,Class,Age
str,str,str
"""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""ANASTACIA OF ASTORA""","""Low""","""Young"""
"""ANDRE OF ASTORA""","""Low""","""Old"""
"""BIG HAТ LOGAN""","""Low""","""Old"""
"""BLACKSMITH VAMOS""","""Low""","""Old"""


In [5]:
# Open raw text to look for missing verbs
with open(data_path + 'Ds corpus input.txt', 'r', encoding='utf-8') as file:
        raw_text = file.read()  # Read the entire content into a single string
        # print(raw_text)

In [6]:
# Clean the raw text to facilitate matching with low_context_name_finder
punctuation = [',','.',';',':','!','?']

raw_text_clean = raw_text.replace('\n',' ').replace('…','...').replace('‘',"'").replace("' ","'").strip()
for punct in punctuation:
        raw_text_clean = raw_text_clean.replace(punct, punct + ' ').replace(punct,'')

raw_text_clean = raw_text_clean.split(' ')

raw_text_clean = [word for word in raw_text_clean if word != '']

raw_text_clean = ' '.join(raw_text_clean)

# raw_text_clean

In [7]:
# Extract the speaking character for each row, turn all KWIC lowercase to avoid redundant values.
verbs_df = verbs_df.with_columns(pl.col('Left').map_elements(lambda x: low_context_name_finder(x, raw_text_clean, 100), 
                                                             return_dtype=pl.Utf8).alias('Character'))\
                         .with_columns(pl.col('KWIC').str.to_lowercase().alias('KWIC'))\
                         .with_columns(pl.col('Left').str.replace_all('<s>','').str.replace_all('</s>','\n'),
                                       pl.col('Right').str.replace_all('<s>','').str.replace_all('</s>','\n'))\

verbs_df

Left,KWIC,Right,Character
str,str,str,str
""" NARRATOR Yes, indeed. The D…","""are""","""corralled and led to the north…","""NARRATOR"""
""" NARRATOR Yes, indeed. The D…","""are""","""locked away, to await the end …","""NARRATOR"""
""". And in this land, the Unde…","""art""","""new. Thou fared well to find…","""ALVINA OF THE DARKROOT WOOD"""
""", Lordran. ALVINA OF THE DAR…","""art""","""a strange one! Nevertheless,…","""ALVINA OF THE DARKROOT WOOD"""
"""new. Thou fared well to find…","""''m""","""Alvina of the Darkroot Wood. …","""ALVINA OF THE DARKROOT WOOD"""
…,…,…,…
"""... Times are grim; the leas…","""''re""","""a persistent one, aren''t you?…","""VINCE OF THOROLUND"""
"""; the least you can do is look…","""are""","""n''t you? Hah hah hah. Hon…","""VINCE OF THOROLUND"""
"""again. What business have yo…","""am""","""Vince of Thorolund. Let''s s…","""VINCE OF THOROLUND"""


In [8]:
verbs_df.filter(pl.col('Character').is_null())#['Left'].to_list()

Left,KWIC,Right,Character
str,str,str,str


In [9]:
#Add character class and age information from master table
verbs_df = verbs_df.join(char_master_df, on='Character', how = 'left')

verbs_df

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str
""" NARRATOR Yes, indeed. The D…","""are""","""corralled and led to the north…","""NARRATOR""",null,null
""" NARRATOR Yes, indeed. The D…","""are""","""locked away, to await the end …","""NARRATOR""",null,null
""". And in this land, the Unde…","""art""","""new. Thou fared well to find…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
""", Lordran. ALVINA OF THE DAR…","""art""","""a strange one! Nevertheless,…","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
"""new. Thou fared well to find…","""''m""","""Alvina of the Darkroot Wood. …","""ALVINA OF THE DARKROOT WOOD""","""High""","""Old"""
…,…,…,…,…,…
"""... Times are grim; the leas…","""''re""","""a persistent one, aren''t you?…","""VINCE OF THOROLUND""","""Low""","""Young"""
"""; the least you can do is look…","""are""","""n''t you? Hah hah hah. Hon…","""VINCE OF THOROLUND""","""Low""","""Young"""
"""again. What business have yo…","""am""","""Vince of Thorolund. Let''s s…","""VINCE OF THOROLUND""","""Low""","""Young"""


In [10]:
verbs_df.filter(pl.col('Character').is_null())

Left,KWIC,Right,Character,Class,Age
str,str,str,str,str,str


In [11]:
# Sanity check for extracted characters
verbs_df.to_pandas()['Character'].unique()

array(['NARRATOR', 'ALVINA OF THE DARKROOT WOOD', 'ANASTACIA OF ASTORA',
       'ANDRE OF ASTORA', 'BIG HAТ LOGAN', 'BLACKSMITH VAMOS',
       'CRESTFALLEN MERCHANT', 'CRESTFALLEN WARRIOR',
       'CROSSBREED PRISCILLA', 'DARK SUN GWYNDOLIN', 'DARKMOON KNIGHTESS',
       'DARKSTALKER KAATHЕ', 'DOMHNALL OF ZENA', 'DUSK OF OOLACILE',
       'EINGYI OF THE GREAT SWAMP', 'ELIZABETH KEEPER OF THE SANCTUARY',
       'GIANT BLACKSMITH', 'GRIGGS OF VINHEIM',
       'GWYNEVERE PRINCESS OF SUNLIGHT', 'HAWKEYE GOUGН',
       'INGWARD KEEPER OF THE SEAL', 'KINGSEEKER FRAMPТ',
       'LAURENTIUS OF THЕ GREAT SWAMP', 'LAUTREC OF CARIM',
       "LORD''S BLADE CIARAN", 'MARVELOUS CHESTER', 'OSCAR OF ASTORA',
       'OSWALD OF CARIM', 'PETRUS OF THOROLUND', 'QUELANA OF IZALITH',
       'RHEA OF THOROLUND', 'RICKERT OF VINHEIM', 'SHIVA OF THE EAST',
       'SIEGLINDE OF CATARINA', 'SIEGMEYER OF CATARINA',
       'SOLAIRE OF ASTORA', 'THE FAIR LADY', 'TRUSTY PATCHES', "I'",
       'UNDEAD MERCHANT (FEMAL

In [12]:
# Save enriched dataframe into a csv file.
verbs_df.write_csv(out_path + 'auxiliary_verbs_analysis.csv', separator= ';')